# Stage 1 zero-Jaccard DPO for BioASQ factoid QA (Qwen2.5-3B)

This notebook trains the established 3B SFT adapter with the strict-equivalence Stage 1 subset whose weak-normalized C3/C1 word-set Jaccard score is exactly zero.

- **Data:** 398 raw `C3 > C1` pairs across 176 training questions
- **Objective:** standard DPO
- **3B learning rate:** `5e-7`
- **DPO beta:** `0.05`
- **Training backpropagation budget:** 2,048 tokens (the established memory-safe 3B setting)
- **Generated-dev evaluation context:** 4,096 tokens
- **Checkpoint selection:** official BioASQ dev MRR
- **Scope:** Stage 1 only

The output directory is separate from all prior 3B and 0.5B experiments. With the 2,048-token backpropagation budget, the current deterministic split is expected to retain 247 training pairs across 113 questions; 48 otherwise valid training pairs exceed that budget. Change `TRAIN_BACKPROP_MAX_LENGTH` to `4096` only if the available GPU memory can support it.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = Path("/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cse_dpo.train_factoid_three_stage_dpo as trainer
trainer = importlib.reload(trainer)

Config = trainer.Config
STAGES = trainer.STAGES
STAGED_DATASET_PRESETS = trainer.STAGED_DATASET_PRESETS
load_stage_rows = trainer.load_stage_rows
run_three_stage = trainer.run_three_stage

MODEL_PRESET = "qwen25_3b"
DATASET_PRESET = "gold_supported_strict_equivalence_v2_1130_stage1_jaccard0"
ZERO_JACCARD_ROOT = Path(STAGED_DATASET_PRESETS[DATASET_PRESET])
SFT_3B = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_3b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr"
TRAIN_BACKPROP_MAX_LENGTH = 2048
OUTPUT_ROOT = PROJECT_ROOT / f"Artifacts/cse_dpo/three_stage_dpo_qwen25_3b_strict_equivalence_v2_1130_stage1_jaccard0_standard_dpo_bp{TRAIN_BACKPROP_MAX_LENGTH}"
STAGE1_DATA = ZERO_JACCARD_ROOT / dict(STAGES)["concept_learning"]

STAGE1_SETTINGS = {
    "objective": "dpo",
    "beta": 0.05,
    "learning_rate": 5e-7,
    "epochs": 8,
    "batch_size": 1,
    "gradient_accumulation": 8,
    "max_length": 4096,
    "train_backprop_max_length": TRAIN_BACKPROP_MAX_LENGTH,
    "optimizer": "AdamW",
    "weight_decay": 0.01,
    "warmup_fraction": 0.05,
    "max_grad_norm": 1.0,
    "lora_dropout": 0.05,
    "append_eos_to_completions": True,
    "allow_truncated_examples": False,
    "drop_truncated_examples": True,
    "eval_every_updates": 10,
    "early_stopping_patience": 8,
    "selection_metric": "dev_mrr",
    "semantic_judge_enabled": False,
    "retention_anchor_weight": 0.0,
    "l2sp_weight": 0.0,
}

CFG = Config(
    model_preset=MODEL_PRESET,
    dataset_preset=DATASET_PRESET,
    staged_root=str(ZERO_JACCARD_ROOT),
    initial_adapter=str(SFT_3B),
    output_root=str(OUTPUT_ROOT),
    smoke_test=False,
    stop_after_stage="concept_learning",
    skip_stages=[],
    objective="dpo",
    beta=0.05,
    learning_rate=5e-7,
    optimizer="AdamW",
    weight_decay=0.01,
    warmup_fraction=0.05,
    lora_dropout=0.05,
    batch_size=1,
    gradient_accumulation=8,
    max_length=4096,
    train_backprop_max_length=TRAIN_BACKPROP_MAX_LENGTH,
    generated_eval_max_seq_length=4096,
    generated_eval_max_new_tokens=64,
    evaluate_generated_dev=True,
    semantic_judge_enabled=False,
    stage_settings={"concept_learning": dict(STAGE1_SETTINGS)},
)

print(json.dumps({
    "model": CFG.model_preset,
    "base_model": CFG.base_model,
    "initial_sft_adapter": CFG.initial_adapter,
    "dataset_preset": CFG.dataset_preset,
    "dataset_root": CFG.staged_root,
    "stage1_data": str(STAGE1_DATA),
    "output_root": CFG.output_root,
    "stop_after_stage": CFG.stop_after_stage,
    "stage1_settings": CFG.stage_settings["concept_learning"],
}, indent=2))


In [ ]:
# Validate all inputs and inspect the raw pair counts before loading the GPU.
assert SFT_3B.exists(), SFT_3B
assert ZERO_JACCARD_ROOT.exists(), ZERO_JACCARD_ROOT
assert STAGE1_DATA.exists(), STAGE1_DATA

stages = load_stage_rows(CFG)
stage1 = stages["concept_learning"]
assert len(stage1) == 398, len(stage1)
assert len({row["question_id"] for row in stage1}) == 176
assert all(row.get("chosen_class") == "C3" and row.get("rejected_class") == "C1" for row in stage1)

print(json.dumps({
    "input_validation": "passed",
    "output_root_already_exists": OUTPUT_ROOT.exists(),
    "stage_1_pairs_before_length_filters": len(stage1),
    "stage_1_questions": len({row["question_id"] for row in stage1}),
    "preference": "C3 > C1",
    "word_jaccard": 0.0,
    "training_backprop_context": CFG.stage_settings["concept_learning"]["train_backprop_max_length"],
    "evaluation_context": CFG.generated_eval_max_seq_length,
    "checkpoint_selection": CFG.stage_settings["concept_learning"]["selection_metric"],
}, indent=2))


In [ ]:
# Start the Stage 1 run. A completed compatible stage is reused automatically on rerun.
results = run_three_stage(CFG)
print(json.dumps(results, indent=2, default=str))


In [ ]:
# Inspect the generated run artifacts after training.
out = Path(CFG.output_root)
print("Output root:", out)
for path in sorted(out.glob("**/manifest.json")):
    print("manifest:", path)
for path in sorted(out.glob("**/data_summary.json")):
    print("data summary:", path)
for path in sorted(out.glob("**/training_curves.png")):
    print("plot:", path)
